# tSCS EMG — single pulse, before vs with lidocaine (P04, 17-07-2026)


> **Run §2 first.** The latencies are picked by hand, once per recording, and saved to
> `results/`. Until a recording has saved picks, the sections below raise
> `FileNotFoundError: No saved latencies for ...` for that file — that is the notebook asking
> you to go and pick them, not a failure of the analysis.

## What a single-pulse file is
One **recruitment sweep**: one stimulation window per intensity (10 → 100 mA in 10 mA steps),
each EMG channel stored as a ±100 ms snippet time-locked to the pulse (t = 0).

## What is measured
- **Latency** — picked **by hand** for every muscle × intensity: the time (ms after the pulse) where
  the response starts. `NaN` = no response. This is the one manual step; picks are saved to
  `results/latency_<file>.csv` and reused from there.
- **Peak-to-peak** — from each picked latency, the window
  `[latency + OFFSET_MS, latency + OFFSET_MS + WINDOW_MS]`; p2p = max − min in it (mV).

## Files
Lidocaine was given at **15:14** (`lidocaine test.xlsx`). Four single-pulse files that day =
**2 electrodes × before/after**:

| | electrode 1 | electrode 3 |
|---|---|---|
| before | `145115` (14:51) | `145242` (14:52) |
| with lidocaine | `160715` (16:07) | `160823` (16:08) |

This notebook compares **electrode 1** — the same pair as the burst analysis. The post sweep
stopped at 80 mA (pre went to 100), so the comparison covers 10–80 mA.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import numpy as np
import matplotlib.pyplot as plt

from functions import (set_style, load_run, pretty, waterfall, waterfall_overlay,
                       latency_picker, save_latency_csv, load_latency_csv)
from functions.quantify import plot_p2p_markers
from functions.average import compare_per_muscle
from functions.compare import compare_p2p_violin, compare_latency_violin
set_style()


## 1 · Config

In [ ]:
D = "tSCS_CHUV_data/17-07-2026/P04tscsHealthy/"
ELECTRODE = 1          # <-- 1 or 3: picks the file pair below

FILES = {   # electrode: (before lidocaine, with lidocaine)
    1: ("Single_Pulse_autosave_20260717_145115_858ms.csv",   # 14:51
        "Single_Pulse_autosave_20260717_160715_411ms.csv"),  # 16:07
    3: ("Single_Pulse_autosave_20260717_145242_160ms.csv",   # 14:52
        "Single_Pulse_autosave_20260717_160823_307ms.csv"),  # 16:08   (latencies not picked yet -> §2)
}
CSV_BEFORE, CSV_AFTER = (D + f for f in FILES[ELECTRODE])

XLIM      = (-20, 80)   # ms shown in the waterfalls / picker
WINDOW_MS = 20.0        # peak-to-peak window length (ms)
OFFSET_MS = 2.0         # ...starting this long after the picked latency (ms)
YLIM_P2P  = (-0.5, 10)  # y-range for the pooled p2p comparison; None = auto
YLIM_LAT  = (0, 30)     # y-range for latency plots (ms)

meta_b, t_b, sig_b = load_run(CSV_BEFORE)
meta_a, t_a, sig_a = load_run(CSV_AFTER)
muscles = [c for c in sig_b if c != "Trigger A" and c in sig_a]
print(f"ELECTRODE {ELECTRODE}")
for tag, m_ in (("BEFORE", meta_b), ("AFTER ", meta_a)):
    print(f"{tag}: electrode {m_[0]['electrode']} | {m_[0]['mode']} | pw {m_[0]['pw_us']} us | "
          f"intensities {[x['amp_ma'] for x in m_]} mA")

def _latfile(csv):
    return f"results/latency_{os.path.splitext(os.path.basename(csv))[0]}.csv"
for csv in (CSV_BEFORE, CSV_AFTER):
    f = _latfile(csv)
    print(("picks saved:  " if os.path.exists(f) else "NO PICKS YET: ") + f)


## 2 · Manual latency picking — one block per condition, do each ONCE

Every figure below reads the picks from `results/latency_<file>.csv`; the config cell above says
which conditions still say **NO PICKS YET**. Each condition has its own three cells:

- **A** — loads that file (and any picks already saved for it). Always safe to run.
- **B** — the picker: **uncomment**, run, click the response onset on every muscle × intensity
  (**Set NaN** = no response · ◀ Prev / Next ▶ · muscle dropdown), then comment it again.
- **C** — saves the picks to that file's own CSV: **uncomment**, run, comment again.

The cells of one block only talk to each other (`picks_<name>`), so running the blocks in any
order, or the rest of the notebook in between, can't mix conditions up. A save is refused if
the picks don't match the file's intensities.

### 2·Before lidocaine

In [ ]:
# ---- Before lidocaine · A: load ----
CSV_before = CSV_BEFORE
meta_before, t_before, sig_before = load_run(CSV_before)
picks_before = load_latency_csv(_latfile(CSV_before)) if os.path.exists(_latfile(CSV_before)) else None
print("Before lidocaine:", CSV_before.split("/")[-1], "|", [m["amp_ma"] for m in meta_before], "mA")
print("existing picks reloaded - continue/correct them" if picks_before else "no picks yet - pick them in B")


In [ ]:
# ---- Before lidocaine · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_before = latency_picker(meta_before, t_before, sig_before, muscles, xlim=XLIM, manual_peaks=picks_before)


In [ ]:
# ---- Before lidocaine · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_before, muscles, CSV_before, meta=meta_before))


### 2·With lidocaine

In [ ]:
# ---- With lidocaine · A: load ----
CSV_after = CSV_AFTER
meta_after, t_after, sig_after = load_run(CSV_after)
picks_after = load_latency_csv(_latfile(CSV_after)) if os.path.exists(_latfile(CSV_after)) else None
print("With lidocaine:", CSV_after.split("/")[-1], "|", [m["amp_ma"] for m in meta_after], "mA")
print("existing picks reloaded - continue/correct them" if picks_after else "no picks yet - pick them in B")


In [ ]:
# ---- With lidocaine · B: PICK (uncomment, run, click through, comment again) ----
# %matplotlib widget
# picks_after = latency_picker(meta_after, t_after, sig_after, muscles, xlim=XLIM, manual_peaks=picks_after)


In [ ]:
# ---- With lidocaine · C: SAVE (uncomment, run, comment again) ----
# plt.close("all"); %matplotlib inline
# print("saved", save_latency_csv(picks_after, muscles, CSV_after, meta=meta_after))


## 3 · Raw traces — waterfalls

One trace per intensity stacked at its amplitude, red = stimulus artifact. **Same gain per muscle
in both** (the first call returns the gains, the second reuses them), so a smaller response
with lidocaine actually draws smaller.

### 3a · Before lidocaine

In [ ]:
gains = waterfall(meta_b, t_b, sig_b, muscles, xlim=XLIM)


### 3b · With lidocaine — same gain

In [ ]:
waterfall(meta_a, t_a, sig_a, muscles, xlim=XLIM, gains=gains);


### 3c · Overlay — before (gray) and with lidocaine (orange) on the same panels, same gain

In [ ]:
waterfall_overlay(meta_b, t_b, sig_b, meta_a, t_a, sig_a, muscles, xlim=XLIM, gains=gains);


## 4 · Check — which peaks the peak-to-peak uses

Per muscle, every intensity, the max (red ▲) and min (blue ▼) inside the window that starts at
the picked latency. If a marker sits on the artifact or misses the wave, fix the pick (§2) or
`WINDOW_MS` / `OFFSET_MS` (§1).

### 4a · Before lidocaine

In [ ]:
picks_b = load_latency_csv(_latfile(CSV_BEFORE))
plot_p2p_markers(meta_b, t_b, sig_b, muscles, picks_b, window_ms=WINDOW_MS, offset_ms=OFFSET_MS)


### 4b · With lidocaine

In [ ]:
picks_a = load_latency_csv(_latfile(CSV_AFTER))
plot_p2p_markers(meta_a, t_a, sig_a, muscles, picks_a, window_ms=WINDOW_MS, offset_ms=OFFSET_MS)


## 5 · Latency — before vs with lidocaine

One panel per muscle, x = intensity. **Gray = before, orange = with lidocaine; ● solid = left
arm, ▲ dashed = right arm.** Gray and orange on top of each other = no lidocaine effect.
Trapezius is right-only.

In [ ]:
compare_per_muscle(CSV_BEFORE, CSV_AFTER, metric="latency", ylim=YLIM_LAT);


## 6 · Peak-to-peak — before vs with lidocaine

Same layout, p2p in mV (window `[latency + OFFSET_MS, + WINDOW_MS]`). y-axis is per muscle —
biceps is ~10× the thenar — so compare gray vs orange within a panel, not heights across panels.

In [ ]:
compare_per_muscle(CSV_BEFORE, CSV_AFTER, metric="p2p", window_ms=WINDOW_MS, offset_ms=OFFSET_MS);


## 7 · All muscles pooled

Per intensity, the distribution over muscles — **gray = before, orange = with lidocaine**.
One figure for peak-to-peak, one for latency.

In [ ]:
compare_p2p_violin(CSV_BEFORE, CSV_AFTER, window_ms=WINDOW_MS, offset_ms=OFFSET_MS, ylim=YLIM_P2P)
compare_latency_violin(CSV_BEFORE, CSV_AFTER, ylim=YLIM_LAT)
